# plan/14 cross-hardware benchmark

Runs the three plan/14 benches — `bench_backend.py`, `bench_forward.py`, `bench_gradient.py` — on whatever hardware the notebook session has attached. Auto-detects CPU / NVIDIA GPU / TPU and installs the matching JAX wheel.

**Target hardware matrix (run once per kind):**
CPU · T4 · L4 · G4 · A100 · H100 · TPU v5e-1 · TPU v6e-1

**What each bench measures:**
- `bench_backend.py` — torch vs `jax.jit` on `batched_expectation_value` (Step 2 gate kernel).
- `bench_forward.py` — end-to-end forward pass (`build_batch` + expectation).
- `bench_gradient.py` — parameter-shift gradient; the Step 4 gate.

Results are written to `/tmp/qiskit_trev_bench_<accelerator>.json` and printed at the end. Download those JSONs from each run to compare across hardware.

In [ ]:
import os, platform, subprocess, sys

BRANCH = 'plan-14-jax-port'
REPO   = 'https://github.com/keunjunpark/qiskit-trev.git'

def _detect():
    # Most reliable: ask a preinstalled JAX what devices it sees. Colab/
    # Kaggle TPU runtimes ship jax[tpu] so this import succeeds there;
    # GPU runtimes do the same with jax[cuda].
    try:
        import jax as _j
        devs = _j.devices()
        kinds = {type(d).__name__.lower() for d in devs}
        if any('tpu' in k for k in kinds):
            return 'tpu'
    except Exception:
        pass
    # TPU env-var fallback (older Colab, some Vertex images).
    if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME') or os.path.exists('/dev/accel0'):
        return 'tpu'
    # NVIDIA GPU — nvidia-smi present.
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                             capture_output=True, text=True, check=True).stdout.strip()
        if out:
            name = out.splitlines()[0].strip()
            return f'gpu:{name}'
    except Exception:
        pass
    return 'cpu'

ACCEL = _detect()
print('Detected accelerator:', ACCEL)
print('Python        :', sys.version.split()[0])
print('Platform      :', platform.platform())

In [ ]:
# Install JAX matching the detected accelerator. Torch is assumed preinstalled
# (Colab/Kaggle ship a working GPU-enabled torch on GPU hosts; CPU/TPU hosts
# get CPU-only torch, which is correct for this comparison).

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

if ACCEL == 'tpu':
    _pip('jax[tpu]', '-f', 'https://storage.googleapis.com/jax-releases/libtpu_releases.html')
elif ACCEL.startswith('gpu'):
    _pip('jax[cuda12]')
else:
    _pip('jax')

import jax
print('jax', jax.__version__, '|', 'devices:', jax.devices(), '|', 'backend:', jax.default_backend())

In [ ]:
# Clone the branch and install the package in editable mode.

import shutil
REPO_DIR = '/tmp/qiskit-trev'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, REPO_DIR], check=True)
_pip('-e', REPO_DIR)

os.chdir(REPO_DIR)
print('HEAD:', subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip())

In [ ]:
# Quick sanity — is torch using the accelerator you expect?
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available(),
      '| device count:', torch.cuda.device_count() if torch.cuda.is_available() else 0)
if torch.cuda.is_available():
    print('torch gpu :', torch.cuda.get_device_name(0))

In [ ]:
import contextlib
import io
import os
import runpy
import sys
import time

# Make sure the in-process interpreter can find qiskit_trev. `pip install -e`
# in cell 3 registers it, but the kernel's sys.path may have been cached
# before that — prepend src/ explicitly to be safe. Also clear any stale
# partial imports from earlier failed attempts.
SRC = os.path.join(REPO_DIR, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
for mod in list(sys.modules):
    if mod.startswith('qiskit_trev'):
        del sys.modules[mod]
import qiskit_trev  # noqa: E402
print('qiskit_trev from:', qiskit_trev.__file__)

# Run each bench in-process. On TPU the device is locked to the kernel's
# PID so a subprocess can't initialise the TPU backend; in-process reuses
# the already-initialised device.
BENCHES = [
    'bench_backend.py',
    'bench_forward.py',
    'bench_gradient.py',
    'bench_gradient_large_chi.py',
]
runs = {}

for bench in BENCHES:
    print(f'\n{"="*60}\n  {bench}\n{"="*60}')
    buf = io.StringIO()
    t0 = time.perf_counter()
    err = None
    try:
        with contextlib.redirect_stdout(buf):
            runpy.run_path(f'{REPO_DIR}/bench/{bench}', run_name='__main__')
    except Exception as e:
        err = repr(e)
    elapsed = time.perf_counter() - t0
    print(buf.getvalue())
    if err:
        print(f'--- ERROR ---\n{err}')
    runs[bench] = {
        'returncode': 1 if err else 0,
        'wall_seconds': round(elapsed, 2),
        'stdout': buf.getvalue(),
        'stderr_tail': err or '',
    }

In [ ]:
# Save a portable summary. Download this JSON from each hardware run to
# build the cross-hardware comparison.

import json

summary = {
    'accelerator': ACCEL,
    'jax_version': jax.__version__,
    'jax_devices': [str(d) for d in jax.devices()],
    'torch_version': torch.__version__,
    'torch_cuda': bool(torch.cuda.is_available()),
    'torch_gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'platform': platform.platform(),
    'branch': BRANCH,
    'head': subprocess.run(['git', 'log', '-1', '--format=%H %s'],
                           capture_output=True, text=True, cwd=REPO_DIR).stdout.strip(),
    'runs': runs,
}

slug = ACCEL.replace(':', '_').replace(' ', '_')
out_path = f'/tmp/qiskit_trev_bench_{slug}.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('Wrote', out_path)
print('\n--- Headline numbers ---')
for bench, r in runs.items():
    # Last non-empty lines tend to be the summary ratios from each bench.
    tail = [ln for ln in r['stdout'].splitlines() if ln.strip()][-6:]
    print(f'\n{bench}:')
    for ln in tail:
        print(' ', ln)

## Next

1. Download `/tmp/qiskit_trev_bench_<accelerator>.json` from this session.
2. Switch the runtime to the next accelerator, restart the notebook, run again.
3. Once you have JSONs from each hardware kind, diff the `bench_gradient.py` medians across rows — that is the Step 4 cross-hardware table.